# NeuroGolf 2026 EDA

This notebook reviews the ARC-style NeuroGolf task data before modeling. It is designed to run inside Kaggle, where the competition or companion public data is attached under `/kaggle/input`. The analysis focuses on task coverage, pair counts, grid geometry, color-token behavior, visual inspection, and first-pass solver buckets.

# 1. Setup and Data Loading

## 1.1 Setup and Display Defaults

The runtime assumptions are intentionally Kaggle-native: import common notebook libraries, standardize plot styling, and define the ARC grid palette used for puzzle rendering.

In [ ]:
from __future__ import annotations

import json
import math
import os
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.colors import BoundaryNorm, ListedColormap

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

PLOT_CMAP = "viridis"
plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ARC_COLORS = [
    "#000000",  # 0 black
    "#0074D9",  # 1 blue
    "#FF4136",  # 2 red
    "#2ECC40",  # 3 green
    "#FFDC00",  # 4 yellow
    "#AAAAAA",  # 5 gray
    "#F012BE",  # 6 magenta
    "#FF851B",  # 7 orange
    "#7FDBFF",  # 8 light blue
    "#870C25",  # 9 maroon
]
ARC_CMAP = ListedColormap(ARC_COLORS)
ARC_NORM = BoundaryNorm(np.arange(-0.5, 10.5, 1), ARC_CMAP.N)


def insight(title: str, bullets: list[str]) -> None:
    clean_bullets = [b for b in bullets if b]
    if clean_bullets:
        display(Markdown("### " + title + "\n" + "\n".join(f"- {b}" for b in clean_bullets)))


## 1.2 Data Discovery

The input scanner avoids hard-coded Kaggle dataset names. It searches `/kaggle/input` first and then falls back to common local folders for occasional offline development.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [Path("../input"), Path("input"), Path("data"), Path("../data")]


def candidate_roots() -> list[Path]:
    roots: list[Path] = []
    if KAGGLE_INPUT.exists():
        roots.extend([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()])
        roots.append(KAGGLE_INPUT)
    roots.extend([p for p in LOCAL_CANDIDATES if p.exists()])
    return roots


def find_json_files() -> list[Path]:
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))


json_files = find_json_files()
print(f"Found {len(json_files):,} JSON files")
for path in json_files[:25]:
    print(path)
if len(json_files) > 25:
    print(f"... {len(json_files) - 25:,} more")

insight(
    "Data Discovery Insights",
    [
        f"Discovered {len(json_files):,} JSON files across {len(candidate_roots()):,} candidate input roots.",
        "If this count is zero on Kaggle, attach the NeuroGolf competition data or the companion public task dataset before continuing.",
        "The first printed paths should confirm whether the notebook is reading per-task files or a combined task dictionary.",
    ],
)


## 1.3 Data Loading

NeuroGolf public data can appear as one JSON file per task or as a combined dictionary. This loader normalizes both layouts into a single `tasks` mapping keyed by task id.

In [ ]:
def is_task_payload(obj: Any) -> bool:
    return isinstance(obj, dict) and "train" in obj and "test" in obj


def normalize_task_id(path: Path, key: str | None = None) -> str:
    if key:
        key = str(key)
        return key[:-5] if key.endswith(".json") else key
    return path.stem


def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            with path.open("r") as f:
                obj = json.load(f)
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[normalize_task_id(path, key)] = value
        elif isinstance(obj, list):
            for idx, value in enumerate(obj, start=1):
                if is_task_payload(value):
                    tasks[f"task{idx:03d}"] = value
    return dict(sorted(tasks.items()))


tasks = load_tasks(json_files)
print(f"Loaded {len(tasks):,} tasks")
display(list(tasks)[:10])

insight(
    "Data Loading Insights",
    [
        f"Loaded {len(tasks):,} normalized ARC-style tasks.",
        "The first ids above are the ids later expected by solver and ONNX-export notebooks.",
        "A large gap between JSON files found and tasks loaded usually means some JSON files are metadata rather than ARC task payloads.",
    ],
)


# 2. Dataset Overview

## 2.1 Task Inventory

The inventory converts nested train/test examples into task-level features: pair counts, grid sizes, observed color sets, and whether training outputs change shape.

In [ ]:
def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    arr = np.asarray(grid)
    return tuple(arr.shape) if arr.ndim == 2 else (0, 0)


def grid_colors(grid: list[list[int]]) -> set[int]:
    arr = np.asarray(grid)
    return set(map(int, np.unique(arr))) if arr.size else set()


def task_summary(task_id: str, task: dict[str, Any]) -> dict[str, Any]:
    train = task.get("train", [])
    test = task.get("test", [])
    pairs = train + test

    input_shapes = [grid_shape(pair["input"]) for pair in pairs if "input" in pair]
    output_shapes = [grid_shape(pair["output"]) for pair in pairs if "output" in pair]
    train_output_shapes = [grid_shape(pair["output"]) for pair in train if "output" in pair]

    input_colors = set().union(*(grid_colors(pair["input"]) for pair in pairs if "input" in pair)) if pairs else set()
    output_colors = set().union(*(grid_colors(pair["output"]) for pair in pairs if "output" in pair)) if output_shapes else set()

    train_shape_changes = [
        grid_shape(pair["input"]) != grid_shape(pair["output"])
        for pair in train
        if "input" in pair and "output" in pair
    ]

    return {
        "task_id": task_id,
        "n_train": len(train),
        "n_test": len(test),
        "has_test_outputs": all("output" in pair for pair in test) if test else False,
        "input_shapes": sorted(set(input_shapes)),
        "output_shapes": sorted(set(output_shapes)),
        "train_output_shapes": sorted(set(train_output_shapes)),
        "n_input_colors": len(input_colors),
        "n_output_colors": len(output_colors),
        "input_colors": tuple(sorted(input_colors)),
        "output_colors": tuple(sorted(output_colors)),
        "shape_changes_in_train": any(train_shape_changes),
        "max_input_area": max((r * c for r, c in input_shapes), default=0),
        "max_output_area": max((r * c for r, c in output_shapes), default=0),
    }


summary_df = pd.DataFrame([task_summary(task_id, task) for task_id, task in tasks.items()])
display(summary_df.head(10))

if summary_df.empty:
    inventory_bullets = ["No task inventory is available yet because no tasks were loaded."]
else:
    inventory_bullets = [
        f"The inventory contains {len(summary_df):,} tasks with a median of {summary_df['n_train'].median():.1f} training examples and {summary_df['n_test'].median():.1f} test cases per task.",
        f"Training examples show shape changes in {summary_df['shape_changes_in_train'].sum():,} tasks ({summary_df['shape_changes_in_train'].mean():.1%}).",
        f"The median task uses {summary_df['n_input_colors'].median():.1f} unique input colors, with a maximum of {summary_df['n_input_colors'].max():,}.",
    ]
insight("Task Inventory Insights", inventory_bullets)


## 2.2 Coverage Checks

The competition submission format expects task ids such as `task001` through `task400`. This check makes missing or non-standard ids visible before downstream solver notebooks rely on them.

In [ ]:
print(summary_df.shape)
if summary_df.empty:
    print("No tasks loaded yet. Attach the Kaggle competition/public data and rerun from the top.")
    coverage_bullets = ["Coverage cannot be assessed until task data is attached."]
else:
    display(summary_df.describe(include="all"))

    expected_ids = {f"task{i:03d}" for i in range(1, 401)}
    observed_ids = set(summary_df["task_id"])
    missing_expected = sorted(expected_ids - observed_ids)
    extra_ids = sorted(observed_ids - expected_ids)

    print(f"Expected-style tasks present: {len(expected_ids & observed_ids):,} / 400")
    print(f"Missing expected ids: {missing_expected[:20]}{' ...' if len(missing_expected) > 20 else ''}")
    print(f"Non-standard ids: {extra_ids[:20]}{' ...' if len(extra_ids) > 20 else ''}")

    coverage_bullets = [
        f"Observed {len(expected_ids & observed_ids):,} of the 400 expected-style task ids.",
        f"There are {len(missing_expected):,} missing expected ids and {len(extra_ids):,} non-standard ids.",
        "Resolve naming or attachment issues here before building per-task ONNX submission artifacts.",
    ]
insight("Coverage Insights", coverage_bullets)


# 3. EDA Findings

## 3.1 Pair Distributions

Pair count distributions estimate how much evidence each task provides. Tasks with very few training examples generally need stronger priors and simpler transformation hypotheses.

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    summary_df["n_train"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=plt.get_cmap(PLOT_CMAP)(0.62))
    axes[0].set_title("Training examples per task")
    axes[0].set_xlabel("n_train")
    axes[0].set_ylabel("task count")

    summary_df["n_test"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color=plt.get_cmap(PLOT_CMAP)(0.82))
    axes[1].set_title("Test cases per task")
    axes[1].set_xlabel("n_test")
    axes[1].set_ylabel("task count")
    plt.tight_layout()

    train_mode = summary_df["n_train"].mode().iloc[0]
    test_mode = summary_df["n_test"].mode().iloc[0]
    pair_bullets = [
        f"The most common evidence pattern is {train_mode} training examples and {test_mode} test cases.",
        f"Training example counts range from {summary_df['n_train'].min():,} to {summary_df['n_train'].max():,}; prioritize robust validation on the low-example end.",
        f"Test case counts range from {summary_df['n_test'].min():,} to {summary_df['n_test'].max():,}, so export logic should support multiple test inputs per task.",
    ]
else:
    pair_bullets = ["Pair distributions are unavailable until tasks are loaded."]
insight("Pair Distribution Insights", pair_bullets)


## 3.2 Grid Geometry

Grid geometry affects solver design and ONNX cost. Same-shape tasks often suit masking or recoloring logic, while shape-changing tasks need resize, crop, tile, object extraction, or construction strategies.

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    summary_df["max_input_area"].hist(ax=axes[0], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.50))
    axes[0].set_title("Max input area")
    axes[0].set_xlabel("cells")
    axes[0].set_ylabel("task count")

    summary_df["max_output_area"].hist(ax=axes[1], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.70))
    axes[1].set_title("Max output area")
    axes[1].set_xlabel("cells")

    summary_df["shape_changes_in_train"].value_counts().sort_index().plot(kind="bar", ax=axes[2], color=[plt.get_cmap(PLOT_CMAP)(0.35), plt.get_cmap(PLOT_CMAP)(0.85)])
    axes[2].set_title("Shape changes in training pairs")
    axes[2].set_xlabel("shape changes")
    axes[2].set_ylabel("task count")
    plt.tight_layout()

    largest_tasks = summary_df.sort_values(["max_input_area", "max_output_area"], ascending=False).head(15)
    display(largest_tasks)

    geometry_bullets = [
        f"Input area ranges from {summary_df['max_input_area'].min():,} to {summary_df['max_input_area'].max():,} cells; output area ranges from {summary_df['max_output_area'].min():,} to {summary_df['max_output_area'].max():,} cells.",
        f"Shape-changing tasks account for {summary_df['shape_changes_in_train'].mean():.1%} of loaded tasks and should be handled as a distinct solver family.",
        f"The largest-grid tasks listed above are useful stress tests for ONNX cost and memory assumptions.",
    ]
else:
    geometry_bullets = ["Grid geometry cannot be summarized until tasks are loaded."]
insight("Grid Geometry Insights", geometry_bullets)


## 3.3 Color Usage

ARC colors are discrete symbols rather than natural-image colors. Frequency and palette-size checks help identify tasks that may be solved by color mapping, background handling, object masks, or simple token substitutions.

In [ ]:
def count_grid_values(tasks: dict[str, dict[str, Any]], split: str, field: str) -> Counter:
    counts: Counter = Counter()
    for task in tasks.values():
        for pair in task.get(split, []):
            if field in pair:
                counts.update(np.asarray(pair[field]).ravel().astype(int).tolist())
    return counts


train_input_color_counts = count_grid_values(tasks, "train", "input")
train_output_color_counts = count_grid_values(tasks, "train", "output")

color_df = pd.DataFrame({
    "color": range(10),
    "train_input_cells": [train_input_color_counts.get(i, 0) for i in range(10)],
    "train_output_cells": [train_output_color_counts.get(i, 0) for i in range(10)],
})
display(color_df)

if not color_df.empty:
    ax = color_df.set_index("color")[["train_input_cells", "train_output_cells"]].plot(kind="bar", figsize=(10, 4), colormap=PLOT_CMAP)
    ax.set_title("ARC token frequency in training pairs")
    ax.set_xlabel("ARC color token")
    ax.set_ylabel("cell count")
    plt.tight_layout()

if not summary_df.empty:
    ax = summary_df["n_input_colors"].value_counts().sort_index().plot(kind="bar", figsize=(8, 4), color=plt.get_cmap(PLOT_CMAP)(0.68))
    ax.set_title("Unique input colors per task")
    ax.set_xlabel("unique colors")
    ax.set_ylabel("task count")
    plt.tight_layout()

    dominant_input = int(color_df.sort_values("train_input_cells", ascending=False).iloc[0]["color"])
    dominant_output = int(color_df.sort_values("train_output_cells", ascending=False).iloc[0]["color"])
    color_bullets = [
        f"Color {dominant_input} is the most frequent training-input token; color {dominant_output} is the most frequent training-output token.",
        f"Palette size ranges from {summary_df['n_input_colors'].min():,} to {summary_df['n_input_colors'].max():,} unique input colors per task.",
        "Compare input versus output frequencies to spot colors commonly introduced, removed, or used as background.",
    ]
else:
    color_bullets = ["Color usage cannot be interpreted until tasks are loaded."]
insight("Color Usage Insights", color_bullets)


## 3.4 Structural Deep Dive

This diagnostic separates tasks by output-shape behavior, area expansion/compression, and multiple-test-case risk. These features are useful for deciding whether a baseline can be task-constant, same-shape, or needs input-conditioned logic.

In [ ]:
if not summary_df.empty:
    deep_df = summary_df.copy()
    deep_df["area_delta"] = deep_df["max_output_area"] - deep_df["max_input_area"]
    deep_df["area_ratio"] = np.where(deep_df["max_input_area"] > 0, deep_df["max_output_area"] / deep_df["max_input_area"], np.nan)
    deep_df["test_case_group"] = np.where(deep_df["n_test"] == 1, "single test case", "multiple test cases")
    deep_df["area_group"] = pd.cut(
        deep_df["area_ratio"],
        bins=[-np.inf, 0.5, 0.95, 1.05, 2.0, np.inf],
        labels=["strong compression", "mild compression", "same area", "mild expansion", "strong expansion"],
    )

    structural_table = pd.crosstab(deep_df["area_group"], deep_df["shape_changes_in_train"], margins=True)
    test_table = deep_df["test_case_group"].value_counts().rename_axis("test_case_group").reset_index(name="task_count")
    display(structural_table)
    display(test_table)
    display(deep_df.sort_values("area_ratio", ascending=False).head(10)[["task_id", "n_train", "n_test", "input_shapes", "output_shapes", "area_ratio", "area_group"]])
    display(deep_df.sort_values("area_ratio", ascending=True).head(10)[["task_id", "n_train", "n_test", "input_shapes", "output_shapes", "area_ratio", "area_group"]])

    single_test_share = (deep_df["n_test"] == 1).mean()
    same_area_share = (deep_df["area_group"] == "same area").mean()
    structural_bullets = [
        f"{single_test_share:.1%} of tasks have exactly one test case, which makes a constant-output ONNX sanity baseline practical for most tasks.",
        f"{same_area_share:.1%} of tasks preserve approximate max area, but area preservation does not guarantee same-shape behavior.",
        "Strong expansion and strong compression tasks should be reviewed separately because they usually require construction, extraction, tiling, or cropping logic rather than simple recoloring.",
    ]
else:
    structural_bullets = ["Structural deep dive requires loaded task summaries."]
insight("Structural Deep Dive Insights", structural_bullets)


## 3.5 Color Delta Deep Dive

Color set differences between inputs and outputs reveal tasks that introduce new tokens, remove tokens, or preserve the same palette while changing geometry or layout.

In [ ]:
if not summary_df.empty:
    color_delta_df = summary_df.copy()
    color_delta_df["introduced_colors"] = color_delta_df.apply(lambda row: tuple(sorted(set(row["output_colors"]) - set(row["input_colors"]))), axis=1)
    color_delta_df["removed_colors"] = color_delta_df.apply(lambda row: tuple(sorted(set(row["input_colors"]) - set(row["output_colors"]))), axis=1)
    color_delta_df["palette_relation"] = np.select(
        [
            color_delta_df["introduced_colors"].map(len).eq(0) & color_delta_df["removed_colors"].map(len).eq(0),
            color_delta_df["introduced_colors"].map(len).gt(0) & color_delta_df["removed_colors"].map(len).eq(0),
            color_delta_df["introduced_colors"].map(len).eq(0) & color_delta_df["removed_colors"].map(len).gt(0),
        ],
        ["same palette", "introduces colors", "removes colors"],
        default="introduces and removes colors",
    )

    palette_table = color_delta_df["palette_relation"].value_counts().rename_axis("palette_relation").reset_index(name="task_count")
    display(palette_table)
    display(color_delta_df.query("palette_relation != 'same palette'").head(20)[["task_id", "input_colors", "output_colors", "introduced_colors", "removed_colors", "shape_changes_in_train"]])

    ax = palette_table.set_index("palette_relation")["task_count"].plot(kind="bar", figsize=(8, 4), color=plt.get_cmap(PLOT_CMAP)(0.72))
    ax.set_title("Input-output palette relation")
    ax.set_xlabel("palette relation")
    ax.set_ylabel("task count")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

    same_palette_share = (color_delta_df["palette_relation"] == "same palette").mean()
    color_delta_bullets = [
        f"{same_palette_share:.1%} of tasks preserve the same observed input/output color set, making geometry and object logic more important than new-color prediction for those tasks.",
        "Tasks that introduce colors are good candidates for rule families involving marking, filling, completion, or derived labels.",
        "Tasks that remove colors often indicate masking, filtering, object selection, or background normalization.",
    ]
else:
    color_delta_bullets = ["Color delta analysis requires loaded task summaries."]
insight("Color Delta Insights", color_delta_bullets)


# 4. Task Review

## 4.1 Visual Inspection Tools

Manual inspection remains essential for ARC-style transformations. The helper below renders train and test examples consistently, while gracefully handling missing test outputs.

In [ ]:
def show_grid(ax: plt.Axes, grid: list[list[int]], title: str) -> None:
    arr = np.asarray(grid)
    ax.imshow(arr, cmap=ARC_CMAP, norm=ARC_NORM)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which="minor", color="#555555", linewidth=0.5, alpha=0.55)
    ax.tick_params(which="both", left=False, bottom=False, labelleft=False, labelbottom=False)


def show_task(task_id: str, max_pairs: int | None = None) -> None:
    task = tasks[task_id]
    rows: list[tuple[str, int, dict[str, Any]]] = []
    for split in ["train", "test"]:
        pairs = task.get(split, [])
        if max_pairs is not None:
            pairs = pairs[:max_pairs]
        rows.extend((split, idx, pair) for idx, pair in enumerate(pairs, start=1))

    n_rows = len(rows)
    n_cols = 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, max(2.2, 2.4 * n_rows)))
    axes = np.asarray(axes).reshape(n_rows, n_cols)
    fig.suptitle(task_id, fontsize=14, y=1.0)

    for row_idx, (split, idx, pair) in enumerate(rows):
        show_grid(axes[row_idx, 0], pair["input"], f"{split} {idx} input")
        if "output" in pair:
            show_grid(axes[row_idx, 1], pair["output"], f"{split} {idx} output")
        else:
            axes[row_idx, 1].axis("off")
            axes[row_idx, 1].set_title(f"{split} {idx} output unavailable", fontsize=10)
    plt.tight_layout()


if tasks:
    first_task_id = next(iter(tasks))
    show_task(first_task_id)
    visual_bullets = [
        f"The first loaded task, `{first_task_id}`, is rendered as a smoke test for the visualization helper.",
        "Use this view to identify object movement, color replacement, cropping, symmetry, tiling, or counting transformations before implementing solver logic.",
    ]
else:
    visual_bullets = ["Visual inspection requires loaded tasks."]
insight("Visual Inspection Insights", visual_bullets)


## 4.2 Diverse Task Samples

The sample set intentionally mixes large grids, many-color tasks, shape-changing tasks, and compact same-shape tasks. This makes browsing more representative than simply looking at the first few task ids.

In [ ]:
sample_ids: list[str] = []
if not summary_df.empty:
    sample_ids.extend(summary_df.sort_values("max_input_area", ascending=False).head(2)["task_id"].tolist())
    sample_ids.extend(summary_df.sort_values("n_input_colors", ascending=False).head(2)["task_id"].tolist())
    sample_ids.extend(summary_df.query("shape_changes_in_train == True").head(2)["task_id"].tolist())
    sample_ids.extend(summary_df.query("shape_changes_in_train == False").sort_values("max_input_area").head(2)["task_id"].tolist())

sample_ids = list(dict.fromkeys(sample_ids))
display(sample_ids)

insight(
    "Sampling Insights",
    [
        f"Selected {len(sample_ids):,} diverse tasks for visual review.",
        "After running on Kaggle, inspect these examples before deciding whether solver buckets need to be refined.",
    ],
)


In [ ]:
for task_id in sample_ids[:8]:
    show_task(task_id, max_pairs=3)

insight(
    "Manual Review Notes",
    [
        "Look for repeated transformation motifs across sampled tasks, then convert those motifs into explicit solver families.",
        "Flag tasks whose outputs cannot be explained by same-shape recoloring, object extraction, or simple geometric construction.",
    ] if sample_ids else ["No sampled tasks are available yet."],
)


# 5. Planning and Artifacts

## 5.1 Baseline Feasibility

Before building solver families, estimate what a submission-format sanity baseline can cover. This is not meant to be the final modeling approach; it checks how much of the dataset can be packaged with simple ONNX output behavior.

In [ ]:
if not summary_df.empty:
    baseline_df = summary_df.copy()
    baseline_df["constant_output_feasible"] = baseline_df["n_test"].eq(1) & baseline_df["has_test_outputs"]
    feasibility_table = baseline_df[["constant_output_feasible", "shape_changes_in_train"]].value_counts().rename_axis(["constant_output_feasible", "shape_changes_in_train"]).reset_index(name="task_count")
    display(feasibility_table)

    feasible_count = int(baseline_df["constant_output_feasible"].sum())
    feasibility_bullets = [
        f"A constant-output ONNX packaging baseline can cover {feasible_count:,} / {len(baseline_df):,} tasks if test outputs are available and each task has one test case.",
        "Multiple-test-case tasks need either input-conditioned selection or a real transformation solver; they should be isolated early in the baseline notebook.",
        "The first baseline should validate file naming, ONNX input/output names, tensor dtypes, zip creation, and optional onnxruntime checks before solver complexity grows.",
    ]
else:
    feasibility_bullets = ["Baseline feasibility requires loaded task summaries."]
insight("Baseline Feasibility Insights", feasibility_bullets)


## 5.2 Solver Planning Buckets

These buckets are deliberately simple. They create a practical bridge from EDA to later solver notebooks, where each bucket can be replaced by measured solver diagnostics and ONNX export checks.

In [ ]:
def assign_bucket(row: pd.Series) -> str:
    if row["shape_changes_in_train"]:
        return "shape-changing"
    if row["n_input_colors"] <= 2:
        return "low-color same-shape"
    if row["max_input_area"] <= 100:
        return "small same-shape"
    return "larger same-shape"


if not summary_df.empty:
    summary_df = summary_df.copy()
    summary_df["eda_bucket"] = summary_df.apply(assign_bucket, axis=1)
    bucket_df = summary_df["eda_bucket"].value_counts().rename_axis("bucket").reset_index(name="task_count")
    display(bucket_df)

    ax = summary_df["eda_bucket"].value_counts().plot(kind="bar", figsize=(9, 4), color=plt.get_cmap(PLOT_CMAP)(0.74))
    ax.set_title("EDA task buckets for solver planning")
    ax.set_xlabel("bucket")
    ax.set_ylabel("task count")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

    display(summary_df.head())
    leading_bucket = bucket_df.iloc[0]
    bucket_bullets = [
        f"The largest provisional bucket is `{leading_bucket['bucket']}` with {int(leading_bucket['task_count']):,} tasks.",
        "Treat same-shape and shape-changing tasks as separate implementation tracks in the next notebook.",
        "Use bucket-level pass rates later to decide which solver families deserve ONNX optimization effort.",
    ]
else:
    bucket_bullets = ["Solver buckets cannot be assigned until task summaries are available."]
insight("Solver Planning Insights", bucket_bullets)


## 5.3 Export EDA Artifacts

The final cell writes compact CSV summaries under `/kaggle/working` on Kaggle. These artifacts keep later notebooks lightweight and provide a stable reference for solver planning.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
if not summary_df.empty:
    summary_path = OUTPUT_DIR / "neurogolf_eda_task_summary.csv"
    color_path = OUTPUT_DIR / "neurogolf_eda_color_counts.csv"
    summary_df.to_csv(summary_path, index=False)
    color_df.to_csv(color_path, index=False)
    print(f"Wrote {summary_path}")
    print(f"Wrote {color_path}")
    export_bullets = [
        f"Saved task-level features to `{summary_path}`.",
        f"Saved color-token counts to `{color_path}`.",
        "Attach these CSVs to downstream Kaggle notebooks when iterating on solver experiments.",
    ]
else:
    export_bullets = ["No CSV artifacts were written because no task summaries are available."]
insight("Export Insights", export_bullets)
